# Homework 11 - Transfer Learning (Domain Adversarial Training)

> Author: Arvin Liu (r09922071@ntu.edu.tw)

若有任何問題，歡迎來信至助教信箱 ntu-ml-2021spring-ta@googlegroups.com

# Readme


這份作業的任務是Transfer Learning中的Domain Adversarial Training。

<img src="https://i.imgur.com/iMVIxCH.png" width="500px">

> 也就是左下角的那一塊。

## Scenario and Why Domain Adversarial Training
你現在有Source Data + label，其中Source Data和Target Data可能有點關係，所以你想要訓練一個model做在Source Data上並Predict在Target Data上。

但這樣有什麼樣的問題? 相信大家學過Anomaly Detection就會知道，如果有data是在Source Data沒有出現過的(或稱Abnormal的)，那麼model大部分都會因為不熟悉這個data而可能亂做一發。 

以下我們將model拆成Feature Extractor(上半部)和Classifier(下半部)來作例子:
<img src="https://i.imgur.com/IL0PxCY.png" width="500px">

整個Model在學習Source Data的時候，Feature Extrator因為看過很多次Source Data，所以所抽取出來的Feature可能就頗具意義，例如像圖上的藍色Distribution，已經將圖片分成各個Cluster，所以這個時候Classifier就可以依照這個Cluster去預測結果。

但是在做Target Data的時候，Feature Extractor會沒看過這樣的Data，導致輸出的Target Feature可能不屬於在Source Feature Distribution上，這樣的Feature給Classifier預測結果顯然就不會做得好。

## Domain Adversarial Training of Nerural Networks (DaNN)
基於如此，是不是只要讓Soucre Data和Target Data經過Feature Extractor都在同個Distribution上，就會做得好了呢? 這就是DaNN的主要核心。

<img src="https://i.imgur.com/vrOE5a6.png" width="500px">

我們追加一個Domain Classifier，在學習的過程中，讓Domain Classifier去判斷經過Feature Extractor後的Feature是源自於哪個domain，讓Feature Extractor學習如何產生Feature以**騙過**Domain Classifier。 持久下來，通常Feature Extractor都會打贏Domain Classifier。(因為Domain Classifier的Input來自於Feature Extractor，而且對Feature Extractor來說Domain&Classification的任務並沒有衝突。)

如此一來，我們就可以確信不管是哪一個Domain，Feature Extractor都會把它產生在同一個Feature Distribution上。

# Data Introduce

這次的任務是Source Data: 真實照片，Target Data: 手畫塗鴉。

我們必須讓model看過真實照片以及標籤，嘗試去預測手畫塗鴉的標籤為何。

資料位於[這裡](https://drive.google.com/open?id=12-07DSquGdzN3JBHBChN4nMo3i8BqTiL)，以下的code分別為下載和觀看這次的資料大概長甚麼樣子。

特別注意一點: **這次的source和target data的圖片都是平衡的，你們可以使用這個資訊做其他事情。**

In [ ]:
path = "/kaggle/input/competitions/ml2021spring-hw11/"
import matplotlib.pyplot as plt


def no_axis_show(img, title="", cmap=None):
    # imshow, and set the interpolation mode to be "nearest"。
    fig = plt.imshow(img, interpolation="nearest", cmap=cmap)
    # do not show the axes in the images.
    fig.axes.get_xaxis().set_visible(False)
    fig.axes.get_yaxis().set_visible(False)
    plt.title(title)


titles = [
    "horse",
    "bed",
    "clock",
    "apple",
    "cat",
    "plane",
    "television",
    "dog",
    "dolphin",
    "spider",
]
plt.figure(figsize=(18, 18))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    fig = no_axis_show(
        plt.imread(path + f"real_or_drawing/train_data/{i}/{500*i}.bmp"), title=titles[i]
    )

In [ ]:
plt.figure(figsize=(18, 18))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    fig = no_axis_show(
        plt.imread(path + "real_or_drawing/test_data/0/" + str(i).rjust(5, "0") + ".bmp")
    )

# Special Domain Knowledge

因為大家塗鴉的時候通常只會畫輪廓，我們可以根據這點將source data做點邊緣偵測處理，讓source data更像target data一點。

## Canny Edge Detection
算法這邊不贅述，只教大家怎麼用。若有興趣歡迎參考wiki或[這裡](https://medium.com/@pomelyu5199/canny-edge-detector-%E5%AF%A6%E4%BD%9C-opencv-f7d1a0a57d19)。

cv2.Canny使用非常方便，只需要兩個參數: low_threshold, high_threshold。

```cv2.Canny(image, low_threshold, high_threshold)```

簡單來說就是當邊緣值超過high_threshold，我們就確定它是edge。如果只有超過low_threshold，那就先判斷一下再決定是不是edge。

以下我們直接拿source data做做看。

In [ ]:
import cv2
import matplotlib.pyplot as plt

titles = [
    "horse",
    "bed",
    "clock",
    "apple",
    "cat",
    "plane",
    "television",
    "dog",
    "dolphin",
    "spider",
]
plt.figure(figsize=(18, 18))

original_img = plt.imread(path + "real_or_drawing/train_data/0/0.bmp")
plt.subplot(1, 5, 1)
no_axis_show(original_img, title="original")

gray_img = cv2.cvtColor(original_img, cv2.COLOR_RGB2GRAY)
plt.subplot(1, 5, 2)
no_axis_show(gray_img, title="gray scale", cmap="gray")

gray_img = cv2.cvtColor(original_img, cv2.COLOR_RGB2GRAY)
plt.subplot(1, 5, 2)
no_axis_show(gray_img, title="gray scale", cmap="gray")

canny_50100 = cv2.Canny(gray_img, 50, 100)
plt.subplot(1, 5, 3)
no_axis_show(canny_50100, title="Canny(50, 100)", cmap="gray")

canny_150200 = cv2.Canny(gray_img, 150, 200)
plt.subplot(1, 5, 4)
no_axis_show(canny_150200, title="Canny(150, 200)", cmap="gray")

canny_250300 = cv2.Canny(gray_img, 250, 300)
plt.subplot(1, 5, 5)
no_axis_show(canny_250300, title="Canny(250, 300)", cmap="gray")

# Data Process

在這裡我故意將data用成可以使用torchvision.ImageFolder的形式，所以只要使用該函式便可以做出一個datasets。

transform的部分請參考以下註解。
<!-- 
#### 一些細節

在一般的版本上，對灰階圖片使用RandomRotation使用```transforms.RandomRotation(15)```即可。但在colab上需要加上```fill=(0,)```才可運行。
在n98上執行需要把```fill=(0,)```拿掉才可運行。 -->


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Function

import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

source_transform = transforms.Compose([
    # Turn RGB to grayscale. (Bacause Canny do not support RGB images.)
    transforms.Grayscale(),
    # cv2 do not support skimage.Image, so we transform it to np.array, 
    # and then adopt cv2.Canny algorithm.
    transforms.Lambda(lambda x: cv2.Canny(np.array(x), 170, 300)),
    # Transform np.array back to the skimage.Image.
    transforms.ToPILImage(),
    # 50% Horizontal Flip. (For Augmentation)
    transforms.RandomHorizontalFlip(),
    # Rotate +- 15 degrees. (For Augmentation), and filled with zero 
    # if there's empty pixel after rotation.
    transforms.RandomRotation(15, fill=(0,)),
    # Transform to tensor for model inputs.
    transforms.ToTensor(),
])
target_transform = transforms.Compose([
    # Turn RGB to grayscale.
    transforms.Grayscale(),
    # Resize: size of source data is 32x32, thus we need to 
    #  enlarge the size of target data from 28x28 to 32x32。
    transforms.Resize((32, 32)),
    # 50% Horizontal Flip. (For Augmentation)
    transforms.RandomHorizontalFlip(),
    # Rotate +- 15 degrees. (For Augmentation), and filled with zero 
    # if there's empty pixel after rotation.
    transforms.RandomRotation(15, fill=(0,)),
    # Transform to tensor for model inputs.
    transforms.ToTensor(),
])

source_dataset = ImageFolder(path + 'real_or_drawing/train_data', transform=source_transform)
target_dataset = ImageFolder(path + 'real_or_drawing/test_data', transform=target_transform)

# num_workers 让 Canny 边缘检测在背景行程平行算，避免 GPU 等数据；
# pin_memory 加速 CPU→GPU 拷贝；persistent_workers 避免每 epoch 重启 worker。
source_dataloader = DataLoader(source_dataset, batch_size=64, shuffle=True,
                               num_workers=4, pin_memory=True, persistent_workers=True)
target_dataloader = DataLoader(target_dataset, batch_size=64, shuffle=True,
                               num_workers=4, pin_memory=True, persistent_workers=True)
test_dataloader = DataLoader(target_dataset, batch_size=128, shuffle=False,
                             num_workers=4, pin_memory=True, persistent_workers=True)

# Model

Feature Extractor: 典型的VGG-like疊法。

Classifier C1 / C2: 兩個結構相同的 MLP，作為 MCD（Maximum Classifier Discrepancy）的雙分類器。不再使用 Domain Classifier，改靠兩個分類器在目標域上的預測差異來做對抗。

相信作業寫到這邊大家對以下的Layer都很熟悉，因此不再贅述。

In [ ]:
class FeatureExtractor(nn.Module):

    def __init__(self):
        super(FeatureExtractor, self).__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, 1, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(256, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(256, 512, 3, 1, 1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
    def forward(self, x):
        x = self.conv(x).squeeze()
        return x

class LabelPredictor(nn.Module):

    def __init__(self):
        super(LabelPredictor, self).__init__()

        self.layer = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),

            nn.Linear(512, 512),
            nn.ReLU(),

            nn.Linear(512, 10),
        )

    def forward(self, h):
        c = self.layer(h)
        return c

class Classifier2(nn.Module):
    """MCD 的第二個分類器 C2，結構與 LabelPredictor 相同、獨立初始化。"""

    def __init__(self):
        super(Classifier2, self).__init__()

        self.layer = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),

            nn.Linear(512, 512),
            nn.ReLU(),

            nn.Linear(512, 10),
        )

    def forward(self, h):
        c = self.layer(h)
        return c

# Pre-processing

這裡我們選用 AdamW（decoupled weight decay）當 Optimizer，並開啟 fp16 混合精度訓練：CUDA 上走 fp16 加速，CPU 自動降級為 fp32，所以本地也能跑 smoke test。

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

feature_extractor = FeatureExtractor().to(device)
label_predictor = LabelPredictor().to(device)      # C1
classifier2 = Classifier2().to(device)              # C2

class_criterion = nn.CrossEntropyLoss()             # MCD 只需要分類損失，不再需要 domain_criterion

# AdamW (decoupled weight decay, 預設 0.01)
optimizer_F = optim.AdamW(feature_extractor.parameters())
optimizer_C1 = optim.AdamW(label_predictor.parameters())
optimizer_C2 = optim.AdamW(classifier2.parameters())

# fp16 mixed precision: GradScaler 在非 CUDA 上自動禁用，CPU 降級為 fp32
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

# Start Training

## 如何實作 MCD（Maximum Classifier Discrepancy）？

與 DANN 不同，這裡不用 Domain Classifier，改用**兩個結構相同、獨立初始化的分類器 C1 / C2**。對抗的核心從"騙過域分類器"變成"操控兩個分類器的預測差異"：

- **Step A（訓練 C1, C2，凍結 F）**：兩個分類器在源域上學分類（CE loss），同時在目標域上**最大化**兩者的預測差異。差異越大，代表該目標樣本越沒被 F 對齊到源域的決策區。
- **Step B（訓練 F，凍結 C1, C2）**：讓 F **最小化**目標域上的預測差異，把目標特徵拉到兩個分類器都同意的區域，也就是源域特徵分布上。

兩步交替，最終 F 會把目標域對齊到源域，C1 即可用作最終預測。論文 Step A 重複 k 次（預設 k=4）、Step B 1 次；本 notebook `num_k` 預設 4，調小可加速但 C1/C2 訓練不充分。

> 論文：[Maximum Classifier Discrepancy for Unsupervised Domain Adaptation](https://arxiv.org/abs/1908.05965)

## 小提醒
* `lamb` 控制 discrepancy 的權重。**MCD 的 discrepancy 量級遠小於 DANN 的 BCE loss，論文用 1.0**——沿用 DANN 的 0.1 會讓對抗項幾乎失效、F 不對齊 target，導致 target 預測退回隨機。
* 因為我們完全沒有 target 的 label，所以結果如何，只好丟 kaggle 看看囉:)?

In [ ]:
def discrepancy(logits1, logits2):
    # L1 distance between two softmax outputs, measures how much C1 and C2 disagree.
    return torch.mean(
        torch.abs(torch.softmax(logits1, dim=1) - torch.softmax(logits2, dim=1))
    )


def train_epoch(source_dataloader, target_dataloader, lamb, num_k=4):
    """
    Args:
      source_dataloader: source data的dataloader
      target_dataloader: target data的dataloader
      lamb: control the balance of classification and discrepancy.
            MCD discrepancy 量级远小于 DANN 的 BCE loss，论文用 1.0。
      num_k: how many times Step A (train C1/C2) repeats before one Step B (train F).
             Paper default k=4. 调小可加速但 C1/C2 训练不充分。
    """

    # C loss: Classifier C1/C2 的 loss（分类 + 最大化差異）
    # F loss: Feature Extractor 的 loss（最小化差異）
    running_C_loss, running_F_loss = 0.0, 0.0
    total_hit, total_num = 0.0, 0.0
    # fp16 autocast only kicks in on CUDA; on CPU it is a no-op.
    use_amp = torch.cuda.is_available()

    for i, ((source_data, source_label), (target_data, _)) in enumerate(
        zip(source_dataloader, target_dataloader)
    ):
        source_data = source_data.to(device)
        source_label = source_label.to(device)
        target_data = target_data.to(device)

        # --- Step A: train C1, C2 (repeat num_k times) ---
        # Maximize discrepancy on target so classifiers disagree on un-aligned samples.
        for _ in range(num_k):
            with torch.autocast("cuda", enabled=use_amp):
                feat_s = feature_extractor(source_data)
                feat_t = feature_extractor(target_data)

                logits1_s = label_predictor(feat_s)
                logits2_s = classifier2(feat_s)
                cls_loss = class_criterion(logits1_s, source_label) + class_criterion(
                    logits2_s, source_label
                )

                logits1_t = label_predictor(feat_t)
                logits2_t = classifier2(feat_t)
                dis = discrepancy(logits1_t, logits2_t)

                loss_C = cls_loss - lamb * dis

            optimizer_C1.zero_grad()
            optimizer_C2.zero_grad()
            scaler.scale(loss_C).backward()
            scaler.step(optimizer_C1)
            scaler.step(optimizer_C2)
            # GradScaler 规定每个 optimizer 在 update() 前只能 step 一次；
            # num_k>1 时循环里会重复 step C1/C2，必须每次循环后 update() 重置状态。
            scaler.update()

        # --- Step B: train F (minimize discrepancy on target) ---
        with torch.autocast("cuda", enabled=use_amp):
            feat_t = feature_extractor(target_data)
            logits1_t = label_predictor(feat_t)
            logits2_t = classifier2(feat_t)
            dis_F = discrepancy(logits1_t, logits2_t)

            loss_F = lamb * dis_F

        optimizer_F.zero_grad()
        scaler.scale(loss_F).backward()
        scaler.step(optimizer_F)

        scaler.update()

        running_C_loss += loss_C.item()
        running_F_loss += loss_F.item()
        total_hit += torch.sum(torch.argmax(logits1_s, dim=1) == source_label).item()
        total_num += source_data.shape[0]
        print(i, end="\r")

    return (
        running_C_loss / (i + 1),
        running_F_loss / (i + 1),
        total_hit / total_num,
    )


# train 200 epochs
for epoch in range(1000):
    # Paper: lamb=1.0, num_k=4. num_k=4 会比 num_k=1 慢约 4 倍，可先用 num_k=1 快速验证 lamb 效果。
    train_C_loss, train_F_loss, train_acc = train_epoch(
        source_dataloader, target_dataloader, lamb=1.0, num_k=4
    )

    torch.save(feature_extractor.state_dict(), f"extractor_model.bin")
    torch.save(label_predictor.state_dict(), f"predictor_model.bin")

    print(
        "epoch {:>3d}: train C loss: {:6.4f}, train F loss: {:6.4f}, acc {:6.4f}".format(
            epoch, train_C_loss, train_F_loss, train_acc
        )
    )

# Inference

就跟前幾次作業一樣。這裡我使用pd來生產csv，因為看起來比較潮(?)

此外，200 epochs的Accuracy可能會不太穩定，可以多丟幾次或train久一點。

In [ ]:
result = []
label_predictor.eval()
feature_extractor.eval()
with torch.autocast("cuda", enabled=torch.cuda.is_available()):
    for i, (test_data, _) in enumerate(test_dataloader):
        test_data = test_data.to(device)

        class_logits = label_predictor(feature_extractor(test_data))

        x = torch.argmax(class_logits, dim=1).cpu().detach().numpy()
        result.append(x)

import pandas as pd

result = np.concatenate(result)

# Generate your submission
df = pd.DataFrame({"id": np.arange(0, len(result)), "label": result})
df.to_csv("DaNN_submission.csv", index=False)

# Training Statistics

- Number of parameters:
  - Feature Extractor: 2, 142, 336
  - Label Predictor (C1): 530, 442
  - Classifier2 (C2): 530, 442

- Simple
 - Training time on colab: ~ 1 hr
- Medium
 - Training time on colab: 2 ~ 4 hr
- Strong
 - Training time on colab: 5 ~ 6 hrs
- Boss
 - **Unmeasurable**

# Learning Curve (Strong Baseline)
* This method is slightly different from colab.

![Loss Curve](https://i.imgur.com/vIujQyo.png)

# Accuracy Curve (Strong Baseline)
* Note that you cannot access testing accuracy. But this plot tells you that even though the model overfits the training data, the testing accuracy is still improving, and that's why you need to train more epochs.

![Acc Curve](https://i.imgur.com/4W1otXG.png)



# Q&A

有任何問題 Domain Adaptation 的問題可以寄信到ntu-ml-2021spring-ta@googlegroups.com。

時間允許的話我會更新在這裡。

# Special Thanks
這次的作業其實是我出在 2019FALL 的 ML Final Project，以下是我認為在 Final Report 不錯的幾組，有興趣的話歡迎大家參考看看。

[NTU_r08942071_太神啦 / 組長: 劉正仁同學](https://drive.google.com/open?id=11uNDcz7_eMS8dMQxvnWsbrdguu9k4c-c)

[NTU_r08921a08_CAT / 組長: 廖子毅同學](https://drive.google.com/open?id=1xIkSs8HAShdcfV1E0NEnf4JDbL7POZTf)
